# Outlier figures

Visualization of the outlier simulation, from the saved checkpoints in `results/outliers_run/`:
the main estimation figure (data | one perturbed GMM | UOTReg barycenter, with an optional fourth
OT panel), and the supplement transport figure (UOTReg vs OT transport plans, 2x3).

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
REPO_ROOT = P.REPO
from uotreg.models import Generator, TransportMaps
from uotreg.data import GaussianLatentSampler
from uotreg.initialization import load_generator_state
from uotreg.device import to_numpy
from uotreg.metrics import mmd_rbf, w2
from uotreg import outlier_sim as sim

OUTDIR  = P.aux_path("results/outliers_run")
GMM_ONE = 3                          # which mixture to show as "one perturbed GMM" (main fig, panel 2)
GMM_SEL = [2, 3, 4]                  # transport-target mixtures (supplement 2x3)
# colours: inlier / outlier cells; UOT / OT barycenter; T(G); one-GMM highlight
INL_C, OUT_C, UOT_C, OT_C, TG_C = "#3b6fb0", "#d1495b", "#7b3fa0", "#e6812e", "#2ca25f"

# ---- TEXT SIZE -------------------------------------------------------------------------------
# The figures go into the draft at `width=\linewidth` (= 6.5in for an 11pt article with 1in
# margins), so LaTeX rescales them by 6.5 / figsize-width and the size the reader sees is
#     printed pt = fontsize x 6.5 / figsize-width
# The 1x4 canvas is 19in, i.e. a 0.34x rescale -- which is why the labels look small on the page.
# FS multiplies every font size below; the per-element sizes are all still written out explicitly
# next to the thing they control, so they can be nudged one at a time.
FS = globals().get("FS", 1.6)
# JCGS: every label / in-figure text must be >= 10pt. Set the defaults here (explicit >=10 below too).
plt.rcParams.update({"font.size": 11 * FS, "axes.titlesize": 11 * FS, "axes.labelsize": 11 * FS,
                     "figure.titlesize": 12 * FS, "xtick.labelsize": 10 * FS,
                     "ytick.labelsize": 10 * FS, "legend.fontsize": 10 * FS})

## Load data, barycenters, maps
`TEMPLATE_MEANS[:4]` are the 4 heavy inlier modes (mass 0.94), `[4:]` the 4 far outlier modes.
Metrics: **contamination** = barycenter mass nearer an outlier mode; **inlier-dist** = mean distance
to the nearest inlier mode; **MMD** = to an inlier-only ideal sample. Lower = more robust.

In [ ]:
X_all, all_data, labels_all = sim.make_data(seed=25)
Z = GaussianLatentSampler(sim.DIM, device="cpu")
d = sim.DIM
INLIER_M  = np.asarray(sim.TEMPLATE_MEANS[:4], float)
OUTLIER_M = np.asarray(sim.TEMPLATE_MEANS[4:], float)


def load_G(path):
    g = Generator(sim.DIM, sim.DIM, 256, 4, 0.05)
    load_generator_state(g, os.path.join(OUTDIR, path), map_location="cpu"); g.eval()
    return g


def load_T(path):
    T = TransportMaps(sim.DIM, sim.N_MIXTURES, 100, 3, dropout=0.05)
    sd = torch.load(os.path.join(OUTDIR, path), map_location="cpu", weights_only=False)
    T.load_state_dict({k.replace("task_nets.", "heads.").replace(".hidden.", ".layers."): v
                       for k, v in sd.items()}); T.eval()
    return T


G_ub, G_b = load_G("G_10_ub_40.pth"), load_G("G_10_b_40.pth")
T_ub, T_b = load_T("T_10_ub_40.pth"), load_T("T_10_b_40.pth")
with torch.no_grad():
    Xg_ub = to_numpy(G_ub(Z.sample(1000))); TX_ub = to_numpy(T_ub(torch.tensor(Xg_ub)))
    Xg_b  = to_numpy(G_b(Z.sample(1000)));  TX_b  = to_numpy(T_b(torch.tensor(Xg_b)))
X_ub, X_b = to_numpy(G_ub(Z.sample(1600))), to_numpy(G_b(Z.sample(1600)))

pca = PCA(n_components=2).fit(np.vstack([X_all, X_ub, X_b]))
P = pca.transform


def _nn(X, M):
    return np.min(np.linalg.norm(X[:, None, :] - M[None], axis=2), axis=1)


def is_outlier(X):
    return _nn(X, OUTLIER_M) < _nn(X, INLIER_M)          # nearer an outlier mode than any inlier mode


def contamination(X):
    return float(is_outlier(X).mean())


def inlier_dist(X):
    return float(_nn(X, INLIER_M).mean())


def inlier_target(n=1600, seed=0):
    rng = np.random.default_rng(seed); w = np.asarray(sim.WEIGHTS[:4]); w = w / w.sum()
    comp = rng.choice(4, n, p=w)
    return (INLIER_M[comp] + rng.standard_normal((n, d)) * np.sqrt(0.5)).astype(np.float32)


TARGET = inlier_target()


def metrics_text(X):
    return f"W2 to inlier ideal = {w2(X, TARGET):.2f}"


for nm, X in [("UOTReg", X_ub), ("OT", X_b)]:
    print(f"{nm:8s} contamination {contamination(X):.3f}  inlier-dist {inlier_dist(X):.2f}  "
          f"MMD {mmd_rbf(X, TARGET):.3f}  W2 {w2(X, TARGET):.2f}")

## Panel drawers (shared by the 1x3 and the 1x4)

In [ ]:
IM_p, OM_p = P(INLIER_M), P(OUTLIER_M)
_rng = np.random.default_rng(0)
_bg = X_all[_rng.choice(len(X_all), 2500, replace=False)]; BG = P(_bg)     # backdrop cells (fewer)
_dsub = _rng.choice(len(X_all), len(X_all) // 4, replace=False)            # data panel: ~1/4 of the cells
A_half = P(X_all[_dsub]); out_half = is_outlier(X_all[_dsub])
_sub = lambda X, n=1000: X[_rng.choice(len(X), min(n, len(X)), replace=False)]   # display subsample


def panel_data(ax):
    ax.scatter(A_half[~out_half, 0], A_half[~out_half, 1], s=5, c=INL_C, alpha=0.35)
    ax.scatter(A_half[out_half, 0], A_half[out_half, 1], s=6, c=OUT_C, alpha=0.55)
    ax.set_title("Data: inlier vs outlier modes", fontsize=10 * FS)
    ax.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=INL_C, mec="none", label="inlier modes (94%)"),
                       Line2D([0], [0], marker="o", ls="", mfc=OUT_C, mec="none", label="outlier modes (6%)")],
              fontsize=10 * FS, loc="upper right")


def panel_one_gmm(ax, idx=GMM_ONE):
    Gi = P(_sub(all_data[idx]))
    ax.scatter(BG[:, 0], BG[:, 1], s=5, c="0.82", alpha=0.5)
    ax.scatter(Gi[:, 0], Gi[:, 1], s=6, c="#5aae61", alpha=0.5)
    ax.set_title("One perturbed GMM", fontsize=10 * FS)
    ax.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc="0.82", mec="none", label="all cells"),
                       Line2D([0], [0], marker="o", ls="", mfc="#5aae61", mec="none", label=f"GMM {idx}")],
              fontsize=10 * FS, loc="upper right")


def panel_bary(ax, X, c, name, leg=None):
    """`leg` = the legend label for the cloud (defaults to the panel name)."""
    Xp = P(_sub(X))                                       # display subsample (metrics still use full X)
    ax.scatter(BG[:, 0], BG[:, 1], s=5, c="0.85", alpha=0.4)
    ax.scatter(Xp[:, 0], Xp[:, 1], s=5, c=c, alpha=0.55)
    ax.set_title(f"{name} barycenter", fontsize=10 * FS)
    ax.text(0.03, 0.12, metrics_text(X), transform=ax.transAxes, fontsize=10 * FS, family="monospace",
            va="bottom", ha="left", bbox=dict(boxstyle="round", fc="white", ec="0.6", alpha=0.85))
    ax.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=c, mec="none", label=leg or name)],
              fontsize=10 * FS, loc="upper right")


LIM = 24
MAIN_TITLE = "Robust barycenter estimation on the outlier simulation (4 inlier + 4 outlier modes, 10-D)"

## Main 1x4: adds the OT panel, so the robustness gap is direct

In [ ]:
fig4, ax = plt.subplots(1, 4, figsize=(19, 5.2), dpi=140)
panel_data(ax[0]); panel_one_gmm(ax[1])
panel_bary(ax[2], X_ub, UOT_C, "UOTReg", leg="UOTReg")
panel_bary(ax[3], X_b, OT_C, "OT", leg="OT barycenter")
for a in ax:
    a.set_xlim(-LIM, LIM); a.set_ylim(-LIM, LIM); a.set_xlabel("PC1")
ax[0].set_ylabel("PC2")
fig4.suptitle(MAIN_TITLE, fontsize=13 * FS, y=0.98)
fig4.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()

## Supplement 2x3: transport plans -- UOTReg (top) vs OT (bottom)

In [ ]:
pca_t = PCA(n_components=2).fit(np.vstack([np.vstack(all_data), Xg_ub, Xg_b,
                                           TX_ub.reshape(-1, d), TX_b.reshape(-1, d)]))
Pt = pca_t.transform
rng = np.random.default_rng(0)
N_PTS, N_ARR, ZOOM = 500, 45, 13            # fewer cells; #arrows; zoom-in limit
GMM_C = "#e8a33d"                            # single distinct target colour (green T(G) reads on top)

figT, axes = plt.subplots(2, len(GMM_SEL), figsize=(4.3 * len(GMM_SEL), 8.2), dpi=130,
                          sharex=True, sharey=True)
for row, (Xg, TX, name) in enumerate([(Xg_ub, TX_ub, "UOTReg"), (Xg_b, TX_b, "OT")]):
    Xg_p = Pt(Xg); gsub = rng.choice(len(Xg), N_PTS, replace=False)
    tgsub = rng.choice(len(Xg), int(0.7 * N_PTS), replace=False)   # T(G): 30% fewer points (AE: less clutter)
    for col, idx in enumerate(GMM_SEL):
        a = axes[row, col]
        Xi = all_data[idx][rng.choice(len(all_data[idx]), N_PTS, replace=False)]
        TXi_p = Pt(TX[:, idx*d:(idx+1)*d])
        a.scatter(Pt(Xi)[:, 0], Pt(Xi)[:, 1], s=8, c=GMM_C, alpha=0.45)      # target GMM
        a.scatter(Xg_p[gsub, 0], Xg_p[gsub, 1], s=7, c=UOT_C, alpha=0.5)     # barycenter G
        a.scatter(TXi_p[tgsub, 0], TXi_p[tgsub, 1], s=7, c=TG_C, alpha=0.6)  # T(G): 30% fewer than G
        for i in rng.choice(len(Xg), N_ARR, replace=False):                  # clearer arrows
            a.annotate("", xy=tuple(TXi_p[i]), xytext=tuple(Xg_p[i]),
                       arrowprops=dict(arrowstyle="->", color="black", lw=1.2, alpha=0.85))
        a.set_title(f"{name} -> GMM {idx}", fontsize=10 * FS)
        a.set_xlim(-ZOOM, ZOOM); a.set_ylim(-ZOOM, ZOOM)
        if col == 0: a.set_ylabel(f"{name}\nPC2", fontsize=10 * FS)
        if row == 1: a.set_xlabel("PC1")
# margins: the header stack is suptitle (2 lines) -> legend row -> panel titles -> panels, so `top`
# has to leave room for all three; left/bottom hold the y ticks + 2-line row label and the x label.
figT.subplots_adjust(top=0.845, bottom=0.085, left=0.075, right=0.98, hspace=0.16, wspace=0.08)
figT.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=UOT_C, mec="none", label="barycenter G"),
                     Line2D([0], [0], marker="o", ls="", mfc=GMM_C, mec="none", label="target GMM"),
                     Line2D([0], [0], marker="o", ls="", mfc=TG_C, mec="none", label="T(G)")],
            loc="upper center", ncol=3, fontsize=10 * FS, frameon=False, bbox_to_anchor=(0.5, 0.925))
figT.suptitle("Transport plans G -> T(G): T(G) matches the target GMM;\n"
              "UOTReg maps (top) more coherent than OT (bottom)",   # 2 lines: one would run off the canvas
              fontsize=11 * FS, y=0.995)
plt.show()